# 04 · Image CNN baseline (optional, for the paper)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AsemaniJoshua/SignLens/blob/main/notebooks/04_image_cnn_baseline.ipynb)

Your old project had an image CNN as its third model. This notebook rebuilds it properly, so you can **compare pixels against landmarks** under the same honest test. That comparison is a useful result for a publication.

Differences from the old `train_model_image.py`:

| old | new |
|---|---|
| whole video frame, so the model could learn the room and lighting | **hand crop** found by MediaPipe, so it sees the hand only |
| small CNN trained from scratch on ~800 near-duplicate frames | **MobileNetV3-Small pretrained on ImageNet**, fine-tuned |
| `validation_split` on shuffled frames of the same clip | grouped split + leave-one-source-out |
| `horizontal_flip` *and* validation from the same augmented generator | augmentation inside the model, training only |

Needs a **GPU** runtime (*Runtime → Change runtime type → T4*) and the Kaggle datasets (same setup as notebook 01). Expect the landmark model (02) to generalize better to new people at a fraction of the size. If it doesn't, that's worth knowing too.

In [ ]:
#@title Setup: get the code and data (run this first)
import os, sys, subprocess

REPO_URL = "https://github.com/AsemaniJoshua/SignLens"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
IN_COLAB = "google.colab" in sys.modules

def sh(cmd):
    """Run a shell command; stop the notebook if it fails."""
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

if IN_COLAB:
    REPO_DIR = "/content/SignLens"
    if not os.path.isdir(REPO_DIR):
        # Shallow + sparse clone: only notebooks/ and the small keypoint files
        # (~225 MB, a few seconds). Skips node_modules, videos and images.
        sh(f"GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 --branch {BRANCH} "
           f"--filter=blob:none --sparse {REPO_URL} {REPO_DIR}")
        sh(f"cd {REPO_DIR} && git sparse-checkout set --no-cone /notebooks/ "
           "/model/dataset/static_keypoints_optimized/ "
           "'/model/dataset/video_keypoints_no_face/*/[0-9].npy' "
           "'/model/initial build/dataset/' /model/model_endpoints/app/")
    sh("pip install -q mediapipe kagglehub")
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/SignLens"   # outputs survive the session
else:
    REPO_DIR = os.environ.get("SIGNLENS_REPO_DIR", os.path.abspath(".."))
    WORK_DIR = os.environ.get("SIGNLENS_WORK_DIR", os.path.join(REPO_DIR, "work"))

MODEL_DIR = os.path.join(REPO_DIR, "model")
DATA_DIR = os.path.join(WORK_DIR, "data")
OUT_DIR = os.path.join(WORK_DIR, "models")
for d in (WORK_DIR, DATA_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, os.path.join(REPO_DIR, "notebooks"))

import numpy as np
import signlens_core as sc
QUICK = os.environ.get("SIGNLENS_QUICK") == "1"   # tiny settings for automated tests
print(f"signlens_core {sc.__version__} | repo data: {MODEL_DIR} | outputs: {WORK_DIR}")

In [ ]:
#@title Options
DATASETS = {"grassknoted/asl-alphabet": 50, "danrasband/asl-alphabet-test": 5, "lexset/synthetic-asl-alphabet": 1}
PER_CLASS = 250  #@param {type:"integer"}
IMG = 160  #@param {type:"integer"}
EPOCHS_HEAD = 8  #@param {type:"integer"}
EPOCHS_FINETUNE = 12  #@param {type:"integer"}
MOTION_LABELS = ["J", "Z"]
SEED = 42
CACHE_DIR = os.path.join(DATA_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)

## 1. Hand crops from each dataset (cached in Drive)

In [ ]:
import cv2, kagglehub
from signlens_core import extract
HAND_MODEL = extract.download_model(os.path.join(WORK_DIR, "hand_landmarker.task"))
try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        os.environ.setdefault(k, userdata.get(k))
except Exception:
    pass

def crops_for(slug, bucket):
    src = slug.split("/")[1][:24]
    cache = os.path.join(CACHE_DIR, f"crops_{src}_{PER_CLASS}_{IMG}.npz")
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=False)
        return d["x"], d["label"].astype(object), d["group"].astype(object), np.full(len(d["x"]), src, dtype=object)
    root = kagglehub.dataset_download(slug)
    xs, labels, groups = [], [], []
    with extract.HandExtractor(HAND_MODEL) as ex:
        for lab, folders in sorted(extract.letter_folders(root).items()):
            if lab in MOTION_LABELS or lab.isdigit():
                continue
            files = [(f, n, r) for f in folders for r, n in enumerate(sorted(os.listdir(f)))
                     if n.lower().endswith(extract.IMAGE_EXTS)]
            pick = np.linspace(0, len(files) - 1, min(PER_CLASS, len(files))).astype(int)
            for k in pick:
                folder, name, rank = files[k]
                bgr = cv2.imread(os.path.join(folder, name))
                if bgr is None:
                    continue
                lm, _, _ = ex.from_image(bgr)
                if lm is None:
                    continue
                xs.append(extract.crop_hand(bgr, lm, IMG)); labels.append(lab)
                groups.append(f"{src}:{lab}:{os.path.basename(folder)}:{rank // bucket}")
            print(f"  {src} {lab}: {sum(1 for l in labels if l == lab)} crops")
    x = np.stack(xs).astype(np.uint8)
    np.savez_compressed(cache, x=x, label=np.array(labels, str), group=np.array(groups, str))
    return x, np.array(labels, dtype=object), np.array(groups, dtype=object), np.full(len(x), src, dtype=object)

parts = [crops_for(s, b) for s, b in DATASETS.items()]
X = np.concatenate([p[0] for p in parts]); L = np.concatenate([p[1] for p in parts])
G = np.concatenate([p[2] for p in parts]); S = np.concatenate([p[3] for p in parts])
LABELS = sc.sort_labels(L); lab2i = {l: i for i, l in enumerate(LABELS)}
Y = np.array([lab2i[l] for l in L])
print(X.shape, "| sources:", dict(zip(*np.unique(S, return_counts=True))))

## 2. Model: MobileNetV3-Small, hand crops only

In [ ]:
import keras, tensorflow as tf
from keras import layers
keras.utils.set_random_seed(SEED)

def build():
    aug = keras.Sequential([layers.RandomRotation(0.06), layers.RandomZoom(0.12),
                            layers.RandomTranslation(0.08, 0.08), layers.RandomContrast(0.3),
                            layers.RandomBrightness(0.25), layers.RandomFlip("horizontal")], name="augment")
    base = keras.applications.MobileNetV3Small(input_shape=(IMG, IMG, 3), include_top=False,
                                               weights="imagenet", include_preprocessing=True)
    base.trainable = False
    inp = keras.Input((IMG, IMG, 3), dtype="uint8")
    x = aug(layers.Rescaling(1.0)(inp))               # augmentation is skipped automatically at inference
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(len(LABELS), activation="softmax")(x)
    return keras.Model(inp, out), base

def fit(tr, va=None, verbose=1):
    model, base = build()
    ds = lambda idx, shuffle: tf.data.Dataset.from_tensor_slices((X[idx], Y[idx])).shuffle(4096 if shuffle else 1, seed=SEED).batch(64).prefetch(2)
    val = ds(va, False) if va is not None and len(va) else None
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(ds(tr, True), validation_data=val, epochs=EPOCHS_HEAD, verbose=verbose)
    base.trainable = True
    for layer in base.layers[:-40]:
        layer.trainable = False                     # fine-tune only the top of the network
    model.compile(optimizer=keras.optimizers.Adam(1e-4), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(ds(tr, True), validation_data=val, epochs=EPOCHS_FINETUNE, verbose=verbose,
              callbacks=[keras.callbacks.EarlyStopping("val_loss", patience=4, restore_best_weights=True)] if val else [])
    return model

tr, va, te = sc.grouped_split(L, G, test_frac=0.2, val_frac=0.1, seed=SEED)
sc.check_no_leak(G, tr, va, te)
cnn = fit(tr, va)
pred = np.array(LABELS)[cnn.predict(X[te], verbose=0).argmax(1)]
sc.evaluate.report(L[te], pred, LABELS, title="CNN on held-out recordings")

## 3. Leave one source out: the fair comparison

Run the same test in notebook 02 on the same sources and put both rows in your results table.

In [ ]:
loso = sc.evaluate.leave_one_source_out(
    S, L, lambda a, b: np.array(LABELS)[fit(a, verbose=0).predict(X[b], verbose=0).argmax(1)])
cnn.save(os.path.join(OUT_DIR, "image_cnn_baseline.keras"))
sc.save_labels(os.path.join(OUT_DIR, "image_cnn_labels.json"), LABELS,
               {"model": "image_cnn_baseline", "input": f"{IMG}x{IMG} RGB hand crop (signlens_core.extract.crop_hand)",
                "heldout_sources": {k: v["accuracy"] for k, v in loso.items()}})